In [12]:
import numpy as np
import pandas as pd

n = 100
# --- random lon/lat ---
lon = np.random.uniform(-6, 36, n)
lat = np.random.uniform(30, 45, n)
# --- random dates in 2024 ---
dates = pd.date_range("2024-01-01", "2024-12-31", freq="D")
time = np.random.choice(dates, n)
# --- set to noon UTC ---
time = pd.to_datetime(time).normalize() + pd.Timedelta(hours=0)
# --- assemble into dataframe ---
df = pd.DataFrame({
    "lon": lon,
    "lat": lat,
    "time": time
})

print(df.head())

         lon        lat       time
0  -1.646186  38.457308 2024-02-26
1  14.915258  38.620727 2024-11-19
2   8.933158  38.480088 2024-04-22
3  23.204887  40.449396 2024-11-10
4  -4.019929  41.885434 2024-10-01


In [2]:
import xarray as xr
ds = xr.open_dataset("fixtures/multi_time.nc")
ds

<xarray.Dataset> Size: 296kB
Dimensions:    (time: 366, latitude: 10, longitude: 10)
Coordinates:
  * time       (time) datetime64[ns] 3kB 2024-01-01 2024-01-02 ... 2024-12-31
  * latitude   (latitude) float32 40B 30.19 31.94 33.69 ... 42.44 44.19 45.98
  * longitude  (longitude) float32 40B -6.0 -1.333 3.375 ... 26.88 31.58 36.29
    depth      float32 4B ...
Data variables:
    uo         (time, latitude, longitude) float32 146kB ...
    vo         (time, latitude, longitude) float32 146kB ...
Attributes:
    title:        Horizontal Velocity (3D) - Daily Mean
    source:       MFS E3R1I
    institution:  Centro Euro-Mediterraneo sui Cambiamenti Climatici - CMCC, ...
    contact:      servicedesk.cmems@mercator-ocean.eu
    comment:      Please check in CMEMS catalogue the INFO section for produc...
    Conventions:  CF-1.0
    references:   Escudier, R., Clementi, E., Omar, M., Cipollone, A., Pistoi...

In [13]:
pts = xr.Dataset(
    {
        "longitude": ("points", df["lon"].to_numpy()),
        "latitude": ("points", df["lat"].to_numpy()),
        "time": ("points", df["time"].to_numpy()),
    }
)

matched = ds.sel(
    longitude=pts["longitude"],
    latitude=pts["latitude"],
    time=pts["time"],
    method="nearest",
)

matched

<xarray.Dataset> Size: 2kB
Dimensions:    (points: 100)
Coordinates:
    depth      float32 4B ...
    latitude   (points) float32 400B 38.94 38.94 38.94 ... 31.94 40.69 44.19
    longitude  (points) float32 400B -1.333 12.79 8.083 ... 3.375 26.88 3.375
    time       (points) datetime64[ns] 800B 2024-02-26 2024-11-19 ... 2024-07-06
Dimensions without coordinates: points
Data variables:
    uo         (points) float32 400B ...
    vo         (points) float32 400B ...
Attributes:
    title:        Horizontal Velocity (3D) - Daily Mean
    source:       MFS E3R1I
    institution:  Centro Euro-Mediterraneo sui Cambiamenti Climatici - CMCC, ...
    contact:      servicedesk.cmems@mercator-ocean.eu
    comment:      Please check in CMEMS catalogue the INFO section for produc...
    Conventions:  CF-1.0
    references:   Escudier, R., Clementi, E., Omar, M., Cipollone, A., Pistoi...

In [14]:
out = xr.Dataset(
    {
        "req_lon": ("points", df["lon"].to_numpy()),
        "req_lat": ("points", df["lat"].to_numpy()),
        "req_time": ("points", df["time"].to_numpy()),
        "match_lon": matched["longitude"],
        "match_lat": matched["latitude"],
        "match_time": matched["time"],
    }
)

In [7]:
out

<xarray.Dataset> Size: 6kB
Dimensions:     (points: 100)
Coordinates:
    depth       float32 4B 1.018
    latitude    (points) float32 400B 31.94 31.94 33.69 ... 37.19 31.94 37.19
    longitude   (points) float32 400B 12.79 12.79 8.083 ... 26.88 3.375 31.58
    time        (points) datetime64[ns] 800B 2024-03-17 ... 2024-12-23
Dimensions without coordinates: points
Data variables:
    req_lon     (points) float64 800B 10.91 14.54 10.34 ... 26.2 3.569 33.71
    req_lat     (points) float64 800B 31.84 31.82 33.72 ... 37.25 32.19 37.64
    req_time    (points) datetime64[ns] 800B 2024-03-16T12:00:00 ... 2024-12-...
    match_lon   (points) float32 400B 12.79 12.79 8.083 ... 26.88 3.375 31.58
    match_lat   (points) float32 400B 31.94 31.94 33.69 ... 37.19 31.94 37.19
    match_time  (points) datetime64[ns] 800B 2024-03-17 ... 2024-12-23

In [15]:
df_out = out.to_dataframe().reset_index()
df_out

,points,req_lon,req_lat,req_time,depth,latitude,longitude,time,match_lon,match_lat,match_time
0,0,-1.646186,38.457308,2024-02-26,1.018237,38.9375,-1.333333,2024-02-26,-1.333333,38.9375,2024-02-26
1,1,14.915258,38.620727,2024-11-19,1.018237,38.9375,12.791667,2024-11-19,12.791667,38.9375,2024-11-19
2,2,8.933158,38.480088,2024-04-22,1.018237,38.9375,8.083333,2024-04-22,8.083333,38.9375,2024-04-22
3,3,23.204887,40.449396,2024-11-10,1.018237,40.6875,22.166666,2024-11-10,22.166666,40.6875,2024-11-10
4,4,-4.019929,41.885434,2024-10-01,1.018237,42.4375,-6.000000,2024-10-01,-6.000000,42.4375,2024-10-01
...,...,...,...,...,...,...,...,...,...,...,...
95,95,13.110226,36.364666,2024-05-15,1.018237,37.1875,12.791667,2024-05-15,12.791667,37.1875,2024-05-15
96,96,-0.368338,34.696612,2024-02-03,1.018237,35.4375,-1.333333,2024-02-03,-1.333333,35.4375,2024-02-03
97,97,3.198413,31.236109,2024-07-29,1.018237,31.9375,3.375000,2024-07-29,3.375000,31.9375,2024-07-29
98,98,25.214290,40.377742,2024-05-13,1.018237,40.6875,26.875000,2024-05-13,26.875000,40.6875,2024-05-13
